<div align="right"><small>
<b>Pablo Emilio Gómez Gómez</b><br>
Proyecto Integrado III - Analítica de Datos<br>
Docente: Andrés Felipe Callejas Jaramillo<br>
Código: PREICA2602B020038<br>
Institución Universitaria Digital de Antioquia<br>
25 de octubre de 2026
</small></div>

# Actividad 1: obtener y preparar datos desde Hugging Face

Este notebook coordina dos clases de Python: una obtiene un dataset público de salud desde Hugging Face y lo convierte en un DataFrame de pandas; la otra recibe el DataFrame ya transformado y genera un resumen en formato `.txt`. El ejercicio se mantiene deliberadamente sencillo para mostrar con claridad la carga, la transformación, la trazabilidad y el control de versiones solicitados en la actividad.

## 1. Importaciones y configuración

Este primer bloque reúne las herramientas que se usarán durante el ejercicio. `pandas` permite transformar los datos; las dos clases se importan desde sus archivos `.py`; y `Path` ayuda a verificar que el resumen final haya quedado guardado.

In [1]:
import pandas as pd
from pathlib import Path

from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

pd.set_option("display.max_columns", 20)

print("Importaciones preparadas correctamente.")

Importaciones preparadas correctamente.


## 2. Carga del dataset desde Hugging Face

La clase recibe tres datos sencillos: el identificador del dataset, la configuración `death` y el split `train`. Toda la conexión con Hugging Face queda encapsulada en `HFDatasetLoader`. El resultado esperado es un DataFrame de **299 filas y 13 columnas**.

In [2]:
cargador = HFDatasetLoader(
    dataset_id="mstz/heart_failure",
    config_name="death",
    split="train",
)

df_original = cargador.cargar()

print(f"Dimensiones originales: {df_original.shape}")
df_original.head()

Dimensiones originales: (299, 13)


,age,has_anaemia,creatinine_phosphokinase_concentration_in_blood,has_diabetes,heart_ejection_fraction,has_high_blood_pressure,platelets_concentration_in_blood,serum_creatinine_concentration_in_blood,serum_sodium_concentration_in_blood,is_male,is_smoker,days_in_study,is_dead
0,75,False,582.0,False,20.0,True,265000.00,1.9,130.0,True,False,4,1
1,55,False,7861.0,False,38.0,False,263358.03,1.1,136.0,True,False,6,1
2,65,False,146.0,False,20.0,False,162000.00,1.3,129.0,True,True,7,1
3,50,True,111.0,False,20.0,False,210000.00,1.9,137.0,True,False,7,1
4,65,True,160.0,True,20.0,False,327000.00,2.7,116.0,False,False,8,1


## 3. Transformación 1: renombrar columnas

Los nombres originales son técnicos y extensos. Se reemplazan por nombres breves en español para que el DataFrame sea más fácil de leer sin alterar los valores.

In [3]:
nombres_columnas = {
    "age": "edad",
    "has_anaemia": "anemia",
    "creatinine_phosphokinase_concentration_in_blood": "cpk",
    "has_diabetes": "diabetes",
    "heart_ejection_fraction": "fraccion_eyeccion",
    "has_high_blood_pressure": "hipertension",
    "platelets_concentration_in_blood": "plaquetas",
    "serum_creatinine_concentration_in_blood": "creatinina_serica",
    "serum_sodium_concentration_in_blood": "sodio_serico",
    "is_male": "sexo_masculino",
    "is_smoker": "fumador",
    "days_in_study": "dias_seguimiento",
    "is_dead": "fallecio",
}

df_transformado = df_original.rename(columns=nombres_columnas).copy()
print("Columnas renombradas:")
print(df_transformado.columns.tolist())

Columnas renombradas:
['edad', 'anemia', 'cpk', 'diabetes', 'fraccion_eyeccion', 'hipertension', 'plaquetas', 'creatinina_serica', 'sodio_serico', 'sexo_masculino', 'fumador', 'dias_seguimiento', 'fallecio']


## 4. Transformación 2: convertir variables booleanas a 0 y 1

Las variables `True/False` se convierten a enteros. Esto deja una representación uniforme para posteriores análisis: `1` indica presencia de la condición registrada y `0` ausencia dentro del dataset.

In [4]:
columnas_booleanas = [
    "anemia",
    "diabetes",
    "hipertension",
    "sexo_masculino",
    "fumador",
]

df_transformado[columnas_booleanas] = (
    df_transformado[columnas_booleanas].astype("int64")
)

df_transformado[columnas_booleanas].head()

,anemia,diabetes,hipertension,sexo_masculino,fumador
0,0,0,1,1,0
1,0,0,0,1,0
2,0,0,0,1,1
3,1,0,0,1,0
4,1,1,0,0,0


## 5. Transformación 3: crear grupos de edad

Se crea `grupo_edad` mediante rangos simples. Esta variable no cambia la edad original; únicamente agrega una clasificación útil para organizar los registros en análisis posteriores.

In [5]:
df_transformado["grupo_edad"] = pd.cut(
    df_transformado["edad"],
    bins=[0, 49, 59, 69, float("inf")],
    labels=["Menor de 50", "50 a 59", "60 a 69", "70 o más"],
    include_lowest=True,
).astype(str)

df_transformado[["edad", "grupo_edad"]].head()

,edad,grupo_edad
0,75,70 o más
1,55,50 a 59
2,65,60 a 69
3,50,50 a 59
4,65,60 a 69


## 6. Transformación 4: crear un indicador derivado

Se añade `seguimiento_mayor_100_dias`. El valor `1` identifica registros con más de 100 días de seguimiento y el valor `0` los demás. Es una condición sencilla creada únicamente para practicar una transformación derivada y conservar una interpretación descriptiva.

In [6]:
df_transformado["seguimiento_mayor_100_dias"] = (
    df_transformado["dias_seguimiento"] > 100
).astype("int64")

df_transformado[
    ["dias_seguimiento", "seguimiento_mayor_100_dias"]
].head()

,dias_seguimiento,seguimiento_mayor_100_dias
0,4,0
1,6,0
2,7,0
3,7,0
4,8,0


## 7. Transformación 5: seleccionar y ordenar el DataFrame final

Para cerrar la preparación, se define explícitamente el orden de las columnas. Así el archivo de resumen trabaja exactamente con la versión final que se observa en el notebook.

In [7]:
columnas_finales = [
    "edad",
    "anemia",
    "cpk",
    "diabetes",
    "fraccion_eyeccion",
    "hipertension",
    "plaquetas",
    "creatinina_serica",
    "sodio_serico",
    "sexo_masculino",
    "fumador",
    "dias_seguimiento",
    "fallecio",
    "grupo_edad",
    "seguimiento_mayor_100_dias",
]

df_transformado = df_transformado[columnas_finales].copy()

print(f"Dimensiones finales: {df_transformado.shape}")
df_transformado.head()

Dimensiones finales: (299, 15)


,edad,anemia,cpk,diabetes,fraccion_eyeccion,hipertension,plaquetas,creatinina_serica,sodio_serico,sexo_masculino,fumador,dias_seguimiento,fallecio,grupo_edad,seguimiento_mayor_100_dias
0,75,0,582.0,0,20.0,1,265000.00,1.9,130.0,1,0,4,1,70 o más,0
1,55,0,7861.0,0,38.0,0,263358.03,1.1,136.0,1,0,6,1,50 a 59,0
2,65,0,146.0,0,20.0,0,162000.00,1.3,129.0,1,1,7,1,60 a 69,0
3,50,1,111.0,0,20.0,0,210000.00,1.9,137.0,1,0,7,1,50 a 59,0
4,65,1,160.0,1,20.0,0,327000.00,2.7,116.0,0,0,8,1,60 a 69,0


## 8. Generación del resumen `.txt`

La segunda clase recibe el DataFrame transformado. El archivo que se genera permite reconocer cuántas filas y columnas tiene la versión final, qué tipo de dato conserva cada variable y cuántos valores nulos aparecen.

In [8]:
resumidor = DatasetSummarizer(df_transformado)
ruta_resumen = resumidor.guardar_resumen("resumen_heart_failure.txt")

print(f"Resumen guardado en: {ruta_resumen.resolve()}")
print()
print(resumidor.construir_resumen())

Resumen guardado en: G:\Mi unidad\15. ING. DE SOFTWARE Y DATOS\05. PROYECTO INTEGRADOR III\ACTIVIDADES\ACTIVIDAD 01\EA1_HeartFailure_Pablo_Gomez\resumen_heart_failure.txt

RESUMEN DEL DATAFRAME TRANSFORMADO
Filas: 299
Columnas: 15
Valores nulos totales: 0

DETALLE DE COLUMNAS
----------------------------------------
1. edad | tipo: int64 | nulos: 0
2. anemia | tipo: int64 | nulos: 0
3. cpk | tipo: float64 | nulos: 0
4. diabetes | tipo: int64 | nulos: 0
5. fraccion_eyeccion | tipo: float64 | nulos: 0
6. hipertension | tipo: int64 | nulos: 0
7. plaquetas | tipo: float64 | nulos: 0
8. creatinina_serica | tipo: float64 | nulos: 0
9. sodio_serico | tipo: float64 | nulos: 0
10. sexo_masculino | tipo: int64 | nulos: 0
11. fumador | tipo: int64 | nulos: 0
12. dias_seguimiento | tipo: int64 | nulos: 0
13. fallecio | tipo: int64 | nulos: 0
14. grupo_edad | tipo: object | nulos: 0
15. seguimiento_mayor_100_dias | tipo: int64 | nulos: 0


## 9. Evidencia final de funcionamiento

Este control final comprueba que el objeto transformado sigue siendo un DataFrame, conserva las 299 filas del dataset, contiene las 15 columnas esperadas y dejó disponible el archivo de resumen.

In [9]:
assert isinstance(df_transformado, pd.DataFrame)
assert df_transformado.shape == (299, 15)
assert Path("resumen_heart_failure.txt").exists()

print("Validación final completada.")
print("- DataFrame transformado: 299 filas x 15 columnas")
print("- Resumen .txt: disponible")
print("- Flujo: carga → transformaciones → resumen")

Validación final completada.
- DataFrame transformado: 299 filas x 15 columnas
- Resumen .txt: disponible
- Flujo: carga → transformaciones → resumen


## Conclusión

La actividad deja un flujo reproducible y modular: la primera clase concentra la conexión y carga desde Hugging Face; el notebook realiza cinco transformaciones verificables; y la segunda clase documenta la estructura del DataFrame final en un archivo `.txt`. Esta separación facilita reutilizar el mismo mecanismo en las próximas actividades del curso y mantiene visible qué responsabilidad corresponde a cada componente.